In [0]:
from pyspark.sql import functions as F

# Lendo as tabelas da camada Silver
df_vra = spark.read.table("voe_bem.silver.vra")

# Exemplo de Agregação Gold: Resumo de Voos por Empresa Aérea e Aeroporto de Origem
df_gold_resumo_voos = (
    df_vra
    .groupBy("ICAO_Empresa_Aérea", "ICAO_Aeródromo_Origem", "Situação_Voo")
    .agg(
        F.count("*").alias("total_voos"),
        # Exemplo de cálculo de atraso em minutos (se chegada real > prevista)
        F.sum(
            F.when(F.col("Chegada_Real") > F.col("Chegada_Prevista"), 1).otherwise(0)
        ).alias("total_voos_atrasados")
    )
    .withColumn("_processing_timestamp", F.current_timestamp())
)

# Salvando na camada Gold
df_gold_resumo_voos.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("voe_bem.golden.resumo_voos_por_empresa")
print("Tabela gold.resumo_voos_por_empresa salva com sucesso!")